In [1]:
# packages and working directory  
import scipy 
import sklearn
import econml 
import arch
import os 
import pandas as pd 
import numpy as np 
import seaborn as sns 
import matplotlib as plt
import statsmodels.api as sm 
from statsmodels.discrete.discrete_model import Probit
from statsmodels.iolib.summary2 import summary_col
import statsmodels.formula.api as smf 
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt 
from scipy import stats
from scipy.stats import ttest_ind
from scipy.optimize import approx_fprime
import biogeme.database as db
import biogeme.biogeme as bio
from biogeme import models
from biogeme import models, database
from biogeme.expressions import Beta, Variable

# Consolidate changing directory and CPI dictionary since these don't change throughout the script
new_directory = r'C:\Users\mrhis\OneDrive\Desktop\PHD\Spain\2021 datasets'
os.chdir(new_directory)



In [2]:
df1 = pd.read_csv('filtered.csv')
def map_lhw(row):
    # Ensure the scenario prefix 'h' is included when constructing the column name
    scenario_prefix = 'h' if not row["scenario"].startswith('h') else ''
    scenario_column_name = f'lhw_{scenario_prefix}{row["scenario"]}'
    return row[scenario_column_name]

# Apply the corrected function
df1['lhw_scenario'] = df1.apply(map_lhw, axis=1)

# Print a sample to verify the column has been created correctly
print(df1[['idperson', 'scenario', 'lhw_scenario']].head())
df1 = df1.sort_values(by='idperson')
df1.head()

   idperson scenario  lhw_scenario
0  87010001       h0             0
1  87060001       h0             0
2  87070001       h0             0
3  87090001       h0             0
4  87170002       h0             0


,idhh,idperson,idmother,idfather,idpartner,idorighh,idorigperson,dag,dgn,dec,...,il_bsarg_64,il_bsarg_70,scenario,original_scenario,choice_made,lhw_h0,lhw_h1,lhw_h2,lhw_h3,lhw_scenario
0,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,0.00,0.00,h0,h2,0,0,7,42,54,0
9160,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,2360.64,2360.64,h2,h2,1,0,7,42,54,42
4580,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,353.44,353.44,h1,h2,0,0,7,42,54,7
13740,870100,87010001,0,0,87010002,870100,87010001,65,1,0,...,3035.10,3035.10,h3,h2,0,0,7,42,54,54
9161,870600,87060001,0,0,0,870600,87060001,32,1,0,...,3043.63,3043.63,h2,h2,1,0,11,44,50,44


In [3]:
# Identify individuals with any negative 'ils_udb_yds' values
negative_c_ids = df1[df1['ils_udb_yds'] <= 250]['idperson'].unique()
negative_l_ids = df1[df1['lhw'] > 80 ]['idperson'].unique()

# Check how many individuals are affected
print(f"Number of individuals with negative leisure: {len(negative_l_ids)}")

# Check how many individuals are affected
print(f"Number of individuals with negative consumption: {len(negative_c_ids)}")
# Filter long_df to exclude all rows belonging to individuals identified in step 1
df1_filt = df1[~df1['idperson'].isin(negative_c_ids)]
df1_filt = df1_filt [~ df1_filt ['idperson'].isin(negative_l_ids)]
# Verify the removal
print(f"Original dataframe size: {df1.shape}")
print(f"Filtered dataframe size: {df1_filt.shape}")


Number of individuals with negative leisure: 3
Number of individuals with negative consumption: 272
Original dataframe size: (18320, 350)
Filtered dataframe size: (17220, 350)


In [4]:
df2 = df1_filt.copy()

In [6]:

# Create the 'labor' column based on the 'scenario' column
df2['labor'] = df2['lhw_scenario']

df2['log_y'] = np.log(df2['ils_udb_yds'])
df2['log_l'] = np.log(80 - df2['labor'])
df2['log2_y'] = df2['log_y']**2 
df2['log2_l'] =  df2['log_l']**2
df2['log_y_l'] = df2['log_y'] * df2['log_l']

In [11]:
import torch
from torch.utils.data import DataLoader
from torch_choice.data import ChoiceDataset, utils
if torch.cuda.is_available():
    print(f'CUDA device used: {torch.cuda.get_device_name()}')
    device = 'cuda'
else:
    print('Running tutorial on CPU.')
    device = 'cpu'
item_index = df2[df2['choice_made'] == 1].sort_values(by='idperson')['scenario'].reset_index(drop=True)
print(item_index)
item_names = ['h0', 'h1', 'h2', 'h3']
num_items = 4
encoder = dict(zip(item_names, range(num_items)))
print(f"{encoder=:}")
item_index = item_index.map(lambda x: encoder[x])
item_index = torch.LongTensor(item_index)
print(f"{item_index=:}")

Running tutorial on CPU.
0       h2
1       h2
2       h0
3       h2
4       h2
        ..
4300    h2
4301    h0
4302    h2
4303    h2
4304    h3
Name: scenario, Length: 4305, dtype: object
encoder={'h0': 0, 'h1': 1, 'h2': 2, 'h3': 3}
item_index=tensor([2, 2, 0,  ..., 2, 2, 3])


In [14]:
# Assuming 'df2' and 'item_index' have been defined and properly prepared
pivot_columns = ['log_y', 'log_l', 'log2_y', 'log2_l', 'log_y_l']
pivoted_data = {}
for column in pivot_columns:
    pivoted_data[column] = utils.pivot3d(df2, dim0='idperson', dim1='scenario', values=column)

# Create the dataset
dataset = ChoiceDataset(
    item_index=item_index,
    **pivoted_data  # Unpacking all pivoted data
).to(device)

# Define the DataLoader
train_dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

# Print the keys of the first batch
for batch in train_dataloader:
    print("Batch keys:", batch.keys())  # Check the first batch keys
    break  # Only inspect the first batch and stop


No `session_index` is provided, assume each choice instance is in its own session.


TypeError: default_collate: batch must contain tensors, numpy arrays, numbers, dicts or lists; found <class 'torch_choice.data.choice_dataset.ChoiceDataset'>

In [15]:
class CustomChoiceDataset(ChoiceDataset):
    def __getitem__(self, index):
        # Here you should retrieve and return a single item from the dataset
        # For instance:
        data = {
            'item_index': self.item_index[index],
            'log_y': self.log_y[index],
            'log_l': self.log_l[index],
            'log2_y': self.log2_y[index],
            'log2_l': self.log2_l[index],
            'log_y_l': self.log_y_l[index]
        }
        return data

# Use the custom dataset class
dataset = CustomChoiceDataset(
    item_index=item_index,
    **pivoted_data
).to(device)

# Define the DataLoader
train_dataloader = DataLoader(dataset, batch_size=10, shuffle=True)

# Test the DataLoader
for batch in train_dataloader:
    print("Batch keys:", batch.keys())  # This should now work correctly
    break


No `session_index` is provided, assume each choice instance is in its own session.
Batch keys: dict_keys(['item_index', 'log_y', 'log_l', 'log2_y', 'log2_l', 'log_y_l'])


In [19]:
def forward(self, batch):
    # Assuming the batch contains all the required keys as tensors
    log_y = batch['log_y']
    log_l = batch['log_l']
    log2_y = batch['log2_y']
    log2_l = batch['log2_l']
    log_y_l = batch['log_y_l']

    # Your forward logic here, using the variables extracted from the batch
    # Example:
    result = some_model_operation(log_y, log_l, log2_y, log2_l, log_y_l)
    return result


In [23]:
# Define model coefficients as constants (consider varying them if needed)
model = ConditionalLogitModel(
    coef_variation_dict={'log_y': 'constant', 'log_l': 'constant', 'log2_y': 'constant', 'log2_l': 'constant', 'log_y_l': 'constant'},
    num_param_dict={'log_y': 1, 'log_l': 1, 'log2_y': 1, 'log2_l': 1, 'log_y_l': 1},
    num_items=num_items
).to(device)


NameError: name 'ConditionalLogitModel' is not defined

In [22]:
for epoch in range(num_epochs):
    for batch in train_dataloader:
        optimizer.zero_grad()
        outputs = model(batch)  # Forward pass
        loss = criterion(outputs, batch['desired_output_key'])  # Compute loss
        loss.backward()  # Backpropagation
        optimizer.step()  # Update model parameters

    print(f'Epoch {epoch+1}, Loss: {loss.item()}')


NameError: name 'num_epochs' is not defined